# 책가방을 매는 AI — RAG 처음부터 만들기
### 두뇌가 모르는 것을, 책가방에서 꺼내 보고 답하게

**나만의 인공지능 — 로컬AI의 정석** · 6강 실습 교재

---

특별편 다마AI에서 봤습니다. **책가방을 매면 알고, 벗으면 모릅니다.**
오늘은 그 책가방을 게임이 아니라 **코드로 직접** 만듭니다.

| | |
|---|---|
| 걸리는 시간 | 20분 (모델 받기 3분) |
| 드는 돈 | **0원** — 두 모델 다 무료 공개 |
| 설치할 것 | 없습니다. 코랩 안에서 끝납니다 |
| 밖으로 나가는 것 | 없습니다. 검색도 생성도 이 안에서 |

> **바탕이 된 논문** — Lewis 외, *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*, Facebook AI, 2020
> arxiv.org/abs/2005.11401 · 「RAG」라는 말이 여기서 처음 나왔습니다.

> ▶️ **런타임 → 런타임 유형 변경 → T4 GPU** 를 켜고 시작하세요. CPU 로도 되지만 느립니다.

---
# 0 · 두뇌는 두 가지 기억을 씁니다

논문의 첫 문장이 이 말입니다. 언어모델에는 기억이 **둘** 있다는 것.

```
   ① 몸에 밴 기억 (parametric)
      학습할 때 두뇌 안에 새겨진 것. 다시 학습하지 않으면 안 바뀝니다.
      → 3강에서 받아 온 모델이 이미 아는 것

   ② 꺼내 보는 기억 (non-parametric)
      밖에 두고, 필요할 때 찾아서 읽는 것.
      → 오늘 만드는 책가방
```

챗GPT가 「작년 12월 우리 회사 매출」을 모르는 건 ①에 없기 때문입니다.
그걸 ①에 넣으려면 다시 학습(파인튜닝)해야 합니다. 비싸고 오래 걸립니다.

**대신 ②에 넣습니다.** 종이에 적어 가방에 넣고, 물어보면 꺼내 보게 합니다.
그것이 **RAG — Retrieval(찾아서) Augmented(보태어) Generation(답한다)** 입니다.

오늘 만들 것은 부품 둘입니다.

| 부품 | 하는 일 | 논문에서 |
|---|---|---|
| **리트리버** | 질문과 비슷한 문서를 찾아온다 | DPR (Dense Passage Retrieval) |
| **생성기** | 찾아온 문서를 읽고 답을 쓴다 | seq2seq 언어모델 |

In [ ]:
!pip install -q sentence-transformers transformers accelerate

import torch
장치 = 'cuda' if torch.cuda.is_available() else 'cpu'
print('장치 :', 장치, '(GPU 가 아니면 런타임 유형을 바꿔 보세요)')
print('준비 끝.')

---
# 1 · 책가방에 넣을 문서

두뇌가 **절대 모르는 것**을 넣어야 실험이 됩니다.
인터넷에 없는 것 — 우리 강의 안의 사실들입니다.

> 이 문서들이 「지식」입니다. 나중에 여러분 회사 매뉴얼, 상품 설명, 강의 노트로 바꾸면 됩니다.

In [ ]:
문서들 = [
    "AI CITY BUILDERS 의 로컬AI 과정은 9월 2일에 시작해 9월 30일에 완결됩니다.",
    "이북 서재는 aicitybuilders.com/ebooks 에 있고, 지금 20권이 있습니다.",
    "수강생 전용 이북은 로그인만 하면 열리고, 멤버십 이북은 유튜브 멤버십 글에 올라온 코드로 엽니다.",
    "5강에서 쓴 음성 모델은 Qwen3-TTS 0.6B 이고, 목소리 지문 파일은 18KB 입니다.",
    "젬마 계열 모델은 파일 도구를 부르지 못하고, 코더 계열 모델이 도구 호출을 합니다.",
    "특별편 다마AI 에서는 책가방을 매면 알고, 벗으면 모릅니다.",
    "안내 인공지능 aicitybuilders.com/ask 는 영상 136편 중 지금 볼 것 두세 편을 골라 줍니다.",
    "질문은 해당 영상의 유튜브 댓글이나 jay@connexionai.kr 로 하면 됩니다.",
    "서울은 대한민국의 수도입니다.",                       # ← 일부러 넣은 상식. 검색이 이걸 안 고르는지 보세요
    "물은 섭씨 100도에서 끓습니다.",
]
print(f'문서 {len(문서들)}개를 책가방에 넣을 준비를 했습니다.')

---
# 2 · 리트리버 — 문장을 숫자로, 뜻이 가까우면 숫자도 가깝게

논문의 리트리버는 **DPR** 입니다. 원리는 하나입니다.

```
   「이북은 어디 있어?」   →  [0.12, -0.83, 0.44, … ]   (숫자 384개)
   「이북 서재는 /ebooks」 →  [0.15, -0.79, 0.41, … ]   ← 가깝다
   「물은 100도에 끓는다」 →  [-0.61, 0.22, -0.90, …]   ← 순서가 뒤로
```

문장을 **뜻이 담긴 숫자 묶음(벡터)** 으로 바꾸는 모델을 씁니다.
숫자 자체는 다 비슷해 보입니다. **누가 더 가까운지 순서**가 요점입니다.
1강에서 글자 하나를 숫자 32개로 바꿨던 그 Embedding 이, 문장 전체로 커진 것입니다.

| | |
|---|---|
| 모델 | `intfloat/multilingual-e5-small` — 한국어 됨 · 118M · MIT |
| 크기 | 470MB |
| 하는 일 | 문장 → 숫자 384개 |

> 논문은 질문용 인코더와 문서용 인코더를 **따로** 두었습니다(bi-encoder).
> e5 는 하나로 쓰되 앞에 `query:` / `passage:` 를 붙여 구분합니다. 같은 생각입니다.

In [ ]:
from sentence_transformers import SentenceTransformer, util

리트리버 = SentenceTransformer('intfloat/multilingual-e5-small', device=장치)

# e5 는 문서엔 passage:, 질문엔 query: 를 붙이라고 합니다 (모델 카드 규칙)
문서벡터 = 리트리버.encode(['passage: ' + d for d in 문서들], normalize_embeddings=True)
print('문서 벡터 모양 :', 문서벡터.shape, '  ← 문서 10개 × 숫자 384개')

# 뜻이 가까운 순서로 줄을 세워 봅니다
a = 리트리버.encode('query: 이북은 어디에 있어?', normalize_embeddings=True)
점수 = util.cos_sim(a, 문서벡터)[0]
print('\n「이북은 어디에 있어?」 와 닮은 순서')
for 순위, i in enumerate(점수.argsort(descending=True), 1):
    print(f'  {순위:2}위 {점수[i].item():.2f}  {문서들[i][:34]}')
print('\n숫자는 다 비슷해 보입니다. 순서를 보세요 — 이북 문서 둘이 1·2위입니다.')
print('상식 문서가 3~6위쯤 붙어 있는데, 작은 모델은 점수가 이렇게 뭉칩니다. 그래서 위에서 두세 개만 씁니다(top-k).')

---
# 3 · 검색해 보기 — 가장 가까운 문서 몇 개

질문도 벡터로 바꾸고, 문서 벡터들과 **얼마나 가까운지** 잽니다. 가까운 순으로 몇 개(top-k)를 꺼냅니다.

이게 다마AI의 **「가방에서 종이를 꺼내는」** 순간입니다.

In [ ]:
def 검색(질문, 개수=3):
    q = 리트리버.encode('query: ' + 질문, normalize_embeddings=True)
    점수 = util.cos_sim(q, 문서벡터)[0]
    순위 = 점수.argsort(descending=True)[:개수]
    return [(문서들[i], 점수[i].item()) for i in 순위]

for 문서, 점 in 검색('젬마로 파일 만들기가 왜 안 돼?'):
    print(f'{점:.2f}  {문서}')

---
# 4 · 생성기 — 읽고 답을 쓰는 두뇌

논문의 생성기는 BART 라는 seq2seq 모델이었습니다.
오늘은 **Qwen2.5-1.5B-Instruct** 를 씁니다. 3강에서 골랐던 것과 같은 계열의, 작은 지시형 모델입니다.

| | |
|---|---|
| 크기 | 3GB (fp16) — 무료 T4 에 넉넉히 올라갑니다 |
| 라이선스 | Apache 2.0 |
| 한국어 | 됩니다 |

> 처음 한 번 3GB 를 받습니다. 2~3분 걸립니다.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

이름 = 'Qwen/Qwen2.5-1.5B-Instruct'
토크나이저 = AutoTokenizer.from_pretrained(이름)
생성기 = AutoModelForCausalLM.from_pretrained(
    이름,
    torch_dtype=torch.float16 if 장치 == 'cuda' else torch.float32,
    device_map=장치,
)

def 답하기(질문, 근거=None, 길이=160):
    if 근거:
        내용 = ('아래 [자료]에 있는 내용만 근거로 한국어로 답해. 자료에 없으면 「자료에 없습니다」라고 해.\n\n'
                '[자료]\n' + '\n'.join('- ' + r for r in 근거) + f'\n\n[질문]\n{질문}')
    else:
        내용 = f'한국어로 짧게 답해.\n\n{질문}'
    대화 = [{'role': 'user', 'content': 내용}]
    # 버전에 따라 텐서를 주기도, 딕셔너리를 주기도 합니다. return_dict=True 로 통일합니다.
    입력 = 토크나이저.apply_chat_template(대화, add_generation_prompt=True,
                                        return_tensors='pt', return_dict=True).to(생성기.device)
    with torch.no_grad():
        출력 = 생성기.generate(**입력, max_new_tokens=길이, do_sample=False,
                             pad_token_id=토크나이저.eos_token_id)
    시작 = 입력['input_ids'].shape[1]
    return 토크나이저.decode(출력[0][시작:], skip_special_tokens=True).strip()

print('생성기 준비 끝.')

---
# 5 · 책가방을 벗고 물어봅니다

먼저 **검색 없이** 생성기에게만 묻습니다. 몸에 밴 기억(①)만으로 답하는 것입니다.

우리 강의 안의 사실은 인터넷에 없으니 **모르거나, 그럴듯하게 지어냅니다.** 그게 정상입니다.

In [ ]:
질문들 = [
    '이북 서재는 어디에 있고 몇 권이야?',
    '젬마로 파일 만들기가 왜 안 돼?',
    '5강 목소리 지문 파일은 몇 KB 야?',
]
for q in 질문들:
    print('Q :', q)
    print('A :', 답하기(q), '\n')

---
# 6 · 책가방을 매고 물어봅니다 — 이것이 RAG

이제 **검색 → 그 문서를 붙여서 → 생성** 순서로 갑니다.

```
   질문 ──▶ 리트리버 ──▶ 가까운 문서 3개 ──▶ 생성기 ──▶ 답
                                          (문서를 읽고 씀)
```

논문의 **RAG-Sequence** 방식입니다. 답 하나를 쓰는 동안 같은 문서 묶음을 계속 봅니다.

In [ ]:
def RAG(질문, 개수=3):
    찾은것 = 검색(질문, 개수)
    근거 = [d for d, _ in 찾은것]
    답 = 답하기(질문, 근거)
    return 답, 찾은것

for q in 질문들:
    답, 찾은것 = RAG(q)
    print('Q :', q)
    print('A :', 답)
    print('   근거 →', ' / '.join(d[:28] + '…' for d, _ in 찾은것), '\n')

---
# 7 · 논문의 두 방식 — RAG-Sequence 와 RAG-Token

| | 어떻게 | 언제 |
|---|---|---|
| **RAG-Sequence** (오늘) | 답 하나에 **같은 문서 묶음** | 답이 한 주제일 때. 대부분 이걸 씁니다 |
| **RAG-Token** | 글자(토큰) **하나하나마다 다른 문서**를 볼 수 있음 | 답이 여러 문서를 오갈 때 |

논문은 둘을 다 만들어 비교했고, 셋 중 두 시험에서 최고 점수를 냈습니다.
지금 세상의 RAG 는 거의 다 Sequence 방식이라, 오늘 만든 것이 표준입니다.

> 다마AI 가 한 일이 정확히 이것이었습니다. 가르치기 = 문서 넣기, 책가방 = 검색, 대답 = 생성.

---
# 8 · 무슨 일이 일어난 것인가

**두뇌를 한 글자도 바꾸지 않았습니다.** 생성기 Qwen 은 그대로입니다.
그런데 5번에서 모르던 것을 6번에서는 압니다. 달라진 것은 **옆에 붙여 준 문서**뿐입니다.

```
   파인튜닝 : 두뇌를 다시 학습  → 비싸고 느림. 지식이 바뀌면 또 학습
   RAG      : 두뇌는 그대로     → 문서만 바꾸면 끝. 오늘 넣고 오늘 씀
```

그래서 회사 매뉴얼, 자주 바뀌는 상품 정보, 개인 노트는 **RAG** 로 갑니다.
말투나 형식처럼 「몸에 배어야 하는 것」만 파인튜닝으로 갑니다. 다마AI EP.2 가 그 얘기였습니다.

### 그런데 문서가 만 개가 되면

지금은 10개라 벡터 비교가 순식간입니다. 만 개, 십만 개가 되면
전부 비교하는 대신 **벡터 저장소(벡터 DB)** 를 씁니다. 원리는 같고, 찾는 속도만 다릅니다.
그 얘기가 다음 시간입니다 — 두뇌 하나 만들어 네 도구에 붙이기.

---
# 9 · 직접 해보기

**① 문서를 여러분 것으로 바꿔 보세요**
1번 셀 `문서들` 에 회사 소개, 상품 설명, 강의 노트를 한 줄씩 넣으세요. **넣은 것만 압니다.**

**② 상식 문서가 어디로 가는지 보세요**
「한국의 수도는?」 하고 물으면 상식 문서가 1위로 올라옵니다. 「이북 어디?」 하면 1·2위에는 절대 못 옵니다. 벡터가 뜻을 담고 있다는 증거입니다.
점수가 서로 붙어 있는 건 작은 모델(118M)의 한계입니다. `bge-m3` 같은 큰 리트리버로 바꾸면 차이가 벌어집니다.

**③ `개수` 를 1 과 10 으로 바꿔 보세요**
1이면 놓치고, 10이면 엉뚱한 것까지 딸려 옵니다. 답이 어떻게 달라지는지 보세요.

**④ 자료에 없는 걸 물어보세요**
「내일 날씨는?」 — 「자료에 없습니다」라고 하는지, 지어내는지 보세요. 지어내면 프롬프트를 더 세게 쓰면 됩니다.

**⑤ 생성기를 바꿔 보세요**
`이름` 을 `Qwen/Qwen2.5-3B-Instruct` 로. 더 똑똑하지만 더 무겁습니다. T4 에서는 3B 가 한계입니다.

---

## 막히면

| 증상 | 이유와 해결 |
|---|---|
| `CUDA out of memory` | 런타임 → 세션 다시 시작. 그래도 나면 생성기를 `Qwen2.5-0.5B-Instruct` 로 |
| 모델 받기가 안 끝남 | 3GB 입니다. 3분은 기다려 주세요 |
| 답이 영어로 나옴 | 프롬프트의 「한국어로」가 빠졌는지 보세요 |
| 검색이 엉뚱한 문서를 뽑음 | 문서를 한 문장씩 짧게. 긴 문서는 쪼개 넣으세요 |
| 그래도 막히면 | 화면을 캡처해 안티그래비티에 붙여 넣고 「이 화면인데 왜 이러냐」고 물으세요 |

---

## 오늘 만든 것

| 부품 | 우리 | 논문 (2020) |
|---|---|---|
| 리트리버 | e5-small · 384차원 | DPR · 768차원 |
| 문서 | 10개 | 위키피디아 2,100만 조각 |
| 생성기 | Qwen2.5-1.5B | BART-large |
| 방식 | RAG-Sequence | RAG-Sequence · RAG-Token |

**하는 일은 똑같습니다.** 크기만 다릅니다.

> 논문 원문 — arxiv.org/pdf/2005.11401
> 리트리버 — huggingface.co/intfloat/multilingual-e5-small
> 생성기 — huggingface.co/Qwen/Qwen2.5-1.5B-Instruct

Connect AI LAB · AI CITY BUILDERS